## Import and Load Dataset

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
RUN_ID = "6f701059-94c7-4969-8ca9-5ba2656995d9"
data_path = Path("../data/interim") / RUN_ID / f"validated_dataset_{RUN_ID}.parquet"

df = pd.read_parquet(data_path)

In [ ]:
df.sample(2)

## Feature eligibility audit

In [ ]:
good_loans = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

bad_loans = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

valid_status = good_loans + bad_loans

df = df[df["loan_status"].isin(valid_status)].copy()

target_map = {status: 0 for status in good_loans}
target_map.update({status: 1 for status in bad_loans})

df["issue_d"] = pd.to_datetime(df["issue_d"])

df.shape

In [ ]:
original_columns = list(df.columns)

print("Total columns:", len(original_columns))
original_columns

In [ ]:
# Identifier Columns
id_cols = [
    "id",
    "member_id",
    "url"
]

df = df.drop(columns=id_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# High Cardinality Text Columns
text_cols = [
    "emp_title",
    "title",
    "desc",
    "zip_code"
]

df = df.drop(columns=text_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Target Leakage Columns
target_cols = [
    "loan_status",
]

df = df.drop(columns=target_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Post-Loan Leakage Columns
leakage_cols = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low"
]

df = df.drop(columns=leakage_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Joint Application Columns
joint_cols = [
    col for col in df.columns if col.startswith("sec_app")
]

joint_cols += [
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint"
]

df = df.drop(columns=joint_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Hardship & Sattlement Columns
hardship_cols = [
    col for col in df.columns if "hardship" in col
]

settlement_cols = [
    col for col in df.columns if "settlement" in col
]

df = df.drop(columns=hardship_cols + settlement_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Constant Columns
constant_cols = [
    col for col in df.columns if df[col].nunique() <= 1
]

print("Constant columns:", constant_cols)
df = df.drop(columns=constant_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# High Missing Columns
missing_ratio = df.isna().mean() * 100
missing_cols = missing_ratio[missing_ratio > 60].index.tolist()

print("High missing columns:", missing_cols)
df = df.drop(columns=missing_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
redundant_cols = [
    "grade",  # Redundant
]

df = df.drop(columns=redundant_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Low Information Columns
low_info_cols = [
    "application_type",        # mostly INDIVIDUAL
    "disbursement_method",     # almost constant
    "initial_list_status",     # weak signal
]

df = df.drop(columns=low_info_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Extra Columns
extra_cols = [
    "funded_amnt",  # redundant with loan_amnt
    "funded_amnt_inv",  # redundant with funded_amnt and loan_amnt
    "num_tl_120dpd_2m",  # weak signal
    "num_tl_30dpd",  # weak signal
    "num_tl_90g_dpd_24m",  # weak signal
    "tot_hi_cred_lim",  # Correlated with total_rev_hi_lim
    "total_il_high_credit_limit",  # Correlated with total_bc_limit
]

df = df.drop(columns=extra_cols)
print("Remaining columns:", df.shape[1])

### Stage-2 Pruning

In [ ]:
# Sparse Behavior Columns
sparse_behavioral_cols = [
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "chargeoff_within_12_mths",
    "delinq_amnt",
    "num_accts_ever_120_pd",
    "num_tl_op_past_12m"
]

df = df.drop(columns=sparse_behavioral_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Account Count Columns
account_count_cols = [
    "num_bc_sats",
    "num_il_tl",
    "num_op_rev_tl",
    "num_rev_accts",
    "num_rev_tl_bal_gt_0",
    "num_sats"
]

df = df.drop(columns=account_count_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Over-Engineered Columns
over_engineered_cols = [
    "avg_cur_bal",
    "tot_coll_amt",
]

df = df.drop(columns=over_engineered_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# High Correlation Time Since Columns
time_since_cols = [
    "mths_since_recent_bc",
    "mo_sin_old_il_acct",
    "mo_sin_old_rev_tl_op",
    "mo_sin_rcnt_rev_tl_op",
]

df = df.drop(columns=time_since_cols)
print("Remaining columns:", df.shape[1])

In [ ]:
# Other Columns
other_cols = [
    "tax_liens",  # weak signal
    "installment",  # Redundant
    "sub_grade",  # Redundant
]

df = df.drop(columns=other_cols)
print("Remaining columns:", df.shape[1])

## Final Columns and Dropped Columns

In [ ]:
remaining_columns = list(df.columns)

print("Remaining columns:", len(remaining_columns))
remaining_columns

In [ ]:
removed_columns = list(set(original_columns) - set(remaining_columns))

print("Removed columns:", len(removed_columns))
removed_columns

## Save Eligible Features

In [ ]:
feature_list = pd.DataFrame({
    "feature_name": remaining_columns
})

feature_list.to_csv(
    "../artifacts/features/eligible_features.csv",
    index=False
)

In [ ]:
# Future use
feature_metadata = pd.DataFrame({
    "feature_name": remaining_columns,
    "type": df[remaining_columns].dtypes.astype(str).values,
    "category": "TBD"
})

feature_metadata